<a href="https://colab.research.google.com/github/backlashblitz/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/backlashblitz/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Data Contract Definition:

One row means: One unique daily performance record for a specific content item (client_id, content_id, report_date).

Time Window: Mid-panel observation month (month=2026-03).

In [1]:
!pip install duckdb

In [2]:
# Setup & Grain Query
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

WAREHOUSE_REL = "hf://datasets/FlyRank/internship-warehouse"

# Query 1: Verify grain (Corrected column names)
q1 = f"""
SELECT client_hash_id, content_hash_id, report_date, COUNT(*)
FROM read_parquet('{WAREHOUSE_REL}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1;
"""
print("Grain violations:", len(con.sql(q1).fetchall()))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain violations: 0


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Categorization

* **Features:** `clicks`, `impressions`, `ctr`, `avg_position`
* **Label / Proxy:** `is_declining_label` (Binary probability of traffic drop over the subsequent observation window)
* **Context:** `client_hash_id`, `content_hash_id`, `report_date`
* **Excluded:** Future performance window metrics (e.g., clicks/impressions from `month=2026-04` onwards) and `_sample` table data during label formulation to prevent temporal target leakage.

In [3]:
# Query 2: Row Count & Date Span
q2 = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT content_hash_id) as unique_content,
    MIN(report_date) as start_date,
    MAX(report_date) as end_date
FROM read_parquet('{WAREHOUSE_REL}/fact_content_daily_performance/month=2026-03/*.parquet');
"""
print("--- Row Count & Date Span ---")
print(con.sql(q2).df())

# Query 3: Availability Check (filtering with IS TRUE)
q3 = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(CASE WHEN client_has_gsc IS TRUE THEN 1 END) as gsc_available_rows,
    ROUND(COUNT(CASE WHEN client_has_gsc IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) as gsc_survival_pct
FROM read_parquet('{WAREHOUSE_REL}/fact_content_daily_performance/month=2026-03/*.parquet');
"""
print("\n--- Availability Check ---")
print(con.sql(q3).df())

--- Row Count & Date Span ---
   total_rows  unique_content start_date   end_date
0     9841378          331437 2026-03-01 2026-03-31

--- Availability Check ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_available_rows  gsc_survival_pct
0     9841378             9841378             100.0


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 5-Feature Frame & Availability Explanations

1. **`feat_clicks_30d`**: Knowable at decision moment because it aggregates historical clicks up to the observation cutoff date (`2026-03-31`).
2. **`feat_impressions_30d`**: Knowable at decision moment because historical impression totals are recorded and finalized during daily batch processing.
3. **`feat_ctr_30d`**: Knowable at decision moment because it is derived strictly from historical 30-day clicks and impressions.
4. **`feat_avg_position_30d`**: Knowable at decision moment because search rank position logs are fully available prior to model execution.
5. **`feat_log_impressions_30d`**: Knowable at decision moment because it applies a deterministic logarithmic transform to knowable historical impressions.

In [7]:
# Build 5-Feature Frame & Spring the Leakage Trap (Corrected Column Names)
q_features = f"""
WITH march_features AS (
    SELECT
        content_hash_id,
        SUM(gsc_clicks) as feat_clicks_30d,
        SUM(gsc_impressions) as feat_impressions_30d,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) as feat_ctr_30d,
        AVG(gsc_avg_position) as feat_avg_position_30d,
        LN(1 + SUM(gsc_impressions)) as feat_log_impressions_30d
    FROM read_parquet('{WAREHOUSE_REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id
),
april_future_target AS (
    -- DELIBERATE LEAK: Querying future month 2026-04 (The Trap)
    SELECT content_hash_id, SUM(gsc_clicks) as LEAKED_future_clicks_april
    FROM read_parquet('{WAREHOUSE_REL}/fact_content_daily_performance/month=2026-04/*.parquet')
    GROUP BY content_hash_id
)
SELECT
    m.*,
    a.LEAKED_future_clicks_april
FROM march_features m
LEFT JOIN april_future_target a ON m.content_hash_id = a.content_hash_id;
"""

# 1. Load dataset containing the deliberate future leakage
df_all = con.sql(q_features).df()
print("--- Dataset WITH Deliberate Leak (Future April Clicks) ---")
print(df_all.head())

# 2. REMOVE THE LEAKED COLUMN to keep the honest feature frame
df_clean = df_all.drop(columns=['LEAKED_future_clicks_april'])
print("\n--- Clean Dataset (Leak Removed Successfully) ---")
print(df_clean.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Dataset WITH Deliberate Leak (Future April Clicks) ---
            content_hash_id  feat_clicks_30d  feat_impressions_30d  \
0  content_7a105f548d9c6916              7.0                6523.0   
1  content_a3ea9792f793ec72              0.0                 453.0   
2  content_36c36abc7650d7af              6.0                5630.0   
3  content_a7da352b73b02668             13.0                4944.0   
4  content_f39be42b42a4e8f6              0.0                  42.0   

   feat_ctr_30d  feat_avg_position_30d  feat_log_impressions_30d  \
0      0.001073               7.209549                  8.783243   
1      0.000000               2.987198                  6.118097   
2      0.001066               6.724039                  8.636042   
3      0.002629               7.244844                  8.506132   
4      0.000000              14.432540                  3.761200   

   LEAKED_future_clicks_april  
0                         8.0  
1                         2.0  
2              

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data Limits & Slice Limitations:

Unbalanced History: Client onboarding dates vary across the panel, meaning older content items have deep historical records while newly tracked URLs have short, truncated timelines.

GSC-Only Early Rows: Early historical partitions contain Google Search Console performance data (gsc_*) but lack integrated Google Analytics 4 (ga4_*) behavioral session tracking, which was attached later.

Window Overlaps: Rolling feature aggregates (e.g., 30-day performance windows) share underlying daily observations when evaluated across adjacent days, introducing temporal correlation between continuous model evaluation snapshots.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- [x] Plain-words data contract defined (Grain, Window, Buckets, Exclusions)
- [x] Three verification queries executed on mid-panel month (`2026-03`)
- [x] Grain verified (0 violations)
- [x] Row count and date span verified (9,841,378 rows from 2026-03-01 to 2026-03-31)
- [x] Availability verified using `IS TRUE` (100% survival rate)
- [x] 5-feature frame built with availability explanations
- [x] Deliberate future leakage trap demonstrated and removed (`LEAKED_future_clicks_april`)
- [x] One named limitation stated